loading the data and prepare the libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVR 
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import f1_score , mean_absolute_error

In [91]:
train_path = Path.cwd() / "Data" /"train.csv"

train_data = pd.read_csv(train_path)

train_data.head()

,recording,day,weekday,unit,minute,minute_of_day,people
0,20260904_1105,2026-09-04,Fri,cam_B,11:16,676,13.103
1,20260904_1105,2026-09-04,Fri,cam_B,11:17,677,17.157
2,20260904_1105,2026-09-04,Fri,cam_B,11:18,678,23.348
3,20260904_1105,2026-09-04,Fri,cam_B,11:19,679,15.478
4,20260904_1105,2026-09-04,Fri,cam_B,11:20,680,16.407


EDA

In [92]:
train_data.info() # inspect the data typer and 

<class 'pandas.DataFrame'>
RangeIndex: 13443 entries, 0 to 13442
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   recording      13443 non-null  str    
 1   day            13443 non-null  str    
 2   weekday        13443 non-null  str    
 3   unit           13443 non-null  str    
 4   minute         13443 non-null  str    
 5   minute_of_day  13443 non-null  int64  
 6   people         13443 non-null  float64
dtypes: float64(1), int64(1), str(5)
memory usage: 735.3 KB


In [93]:
train_data.isnull().sum() # checking for the null values 

recording        0
day              0
weekday          0
unit             0
minute           0
minute_of_day    0
people           0
dtype: int64

In [94]:
train_data.describe()

,minute_of_day,people
count,13443.000000,13443.000000
mean,908.839768,81.575230
std,243.275264,111.203073
min,266.000000,1.119000
25%,753.000000,12.322000
50%,903.000000,28.575000
75%,1122.000000,120.333000
max,1304.000000,998.501000


Feature Engineering

In [95]:

df = train_data.copy()

# Convert date and time columns
df["day"] = pd.to_datetime(df["day"])
df["minute"] = pd.to_datetime(df["minute"], format="%H:%M")

# Extract time features
df["hour"] = df["minute"].dt.hour
df["minute_component"] = df["minute"].dt.minute
df["day_of_month"] = df["day"].dt.day
df["month"] = df["day"].dt.month
df["day_of_week"] = df["day"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Create minute_of_day 
df["minute_of_day"] = df["hour"] * 60 + df["minute_component"]

# Cyclical time features
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["minute_sin"] = np.sin(2 * np.pi * df["minute_of_day"] / 1440)
df["minute_cos"] = np.cos(2 * np.pi * df["minute_of_day"] / 1440)

# Sort chronologically
df = df.sort_values(["unit", "recording", "minute_of_day"])

# Historical features
group = df.groupby(["unit", "recording"])["people"]

for lag in [1, 2, 5, 10, 15, 30, 60]:
    df[f"people_lag_{lag}"] = group.shift(lag)

for window in [5, 15, 30]:
    df[f"people_rolling_mean_{window}"] = (
        group.transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

    df[f"people_rolling_std_{window}"] = (  
        group.transform(  
            lambda x: x.shift(1).rolling(window).std()  
        )  
    )

# Changes in people count
df["people_diff_1"] = group.diff(1)
df["people_diff_5"] = group.diff(5)

# Target: people count in the next minute

target_cols = []
for i in range(1, 16):
    col_name = f"target_{i}"
    df[col_name] = group.shift(-i)
    target_cols.append(col_name)

# Remove rows without sufficient history or missing future 15-min targets
df = df.dropna().reset_index(drop=True)
df.head()

C:\Users\dell\AppData\Local\Temp\ipykernel_12256\2600281205.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["day_of_week"] = df["day"].dt.dayofweek


,recording,day,weekday,unit,minute,minute_of_day,people,hour,minute_component,day_of_month,...,target_6,target_7,target_8,target_9,target_10,target_11,target_12,target_13,target_14,target_15
0,20260904_1105,2026-09-04,Fri,cam_A,1900-01-01 12:05:00,725,53.537,12,5,4,...,52.222,55.030,52.596,52.390,54.965,45.077,37.070,49.788,69.777,64.358
1,20260904_1105,2026-09-04,Fri,cam_A,1900-01-01 12:06:00,726,61.029,12,6,4,...,55.030,52.596,52.390,54.965,45.077,37.070,49.788,69.777,64.358,64.553
2,20260904_1105,2026-09-04,Fri,cam_A,1900-01-01 12:07:00,727,75.211,12,7,4,...,52.596,52.390,54.965,45.077,37.070,49.788,69.777,64.358,64.553,57.473
3,20260904_1105,2026-09-04,Fri,cam_A,1900-01-01 12:08:00,728,78.448,12,8,4,...,52.390,54.965,45.077,37.070,49.788,69.777,64.358,64.553,57.473,44.549
4,20260904_1105,2026-09-04,Fri,cam_A,1900-01-01 12:09:00,729,61.752,12,9,4,...,54.965,45.077,37.070,49.788,69.777,64.358,64.553,57.473,44.549,52.038


In [96]:
# Date features
df["year"] = df["day"].dt.year
#df["month"] = df["day"].dt.month
#df["day_of_week"] = df["day"].dt.dayofweek
cols_to_drop = ["day", "minute", "recording", "minute_of_day"]


# Time features
#df["hour"] = df["minute"].dt.hour
#df["minute_component"] = df["minute"].dt.minute

In [97]:
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
df.head()

C:\Users\dell\AppData\Local\Temp\ipykernel_12256\1424864730.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)


,weekday,unit,people,hour,minute_component,day_of_month,month,day_of_week,is_weekend,hour_sin,...,target_7,target_8,target_9,target_10,target_11,target_12,target_13,target_14,target_15,year
0,Fri,cam_A,53.537,12,5,4,9,4,0,1.224647e-16,...,55.030,52.596,52.390,54.965,45.077,37.070,49.788,69.777,64.358,2026
1,Fri,cam_A,61.029,12,6,4,9,4,0,1.224647e-16,...,52.596,52.390,54.965,45.077,37.070,49.788,69.777,64.358,64.553,2026
2,Fri,cam_A,75.211,12,7,4,9,4,0,1.224647e-16,...,52.390,54.965,45.077,37.070,49.788,69.777,64.358,64.553,57.473,2026
3,Fri,cam_A,78.448,12,8,4,9,4,0,1.224647e-16,...,54.965,45.077,37.070,49.788,69.777,64.358,64.553,57.473,44.549,2026
4,Fri,cam_A,61.752,12,9,4,9,4,0,1.224647e-16,...,45.077,37.070,49.788,69.777,64.358,64.553,57.473,44.549,52.038,2026


In [98]:
# Apply One-Hot Encoding to categorical features
# Convert categorical variables into numerical binary columns

# df = pd.get_dummies(
  #  df,
   # columns=['unit', 'day_of_week'],
  #  dtype=int
#)

# Display the dataset after encoding
# df.head() 

In [99]:
# Separate features (X) and 15 targets (y)
X_train = df.drop(columns=target_cols)
y_train = df[target_cols]

In [100]:
# Manually define categorical features
categorical_features = ['unit', 'day_of_week']

In [101]:
# Automatically identify numerical features 
# (Excluding categorical, binary, and cyclical features)
numerical_features = [
    col for col in X_train.columns 
    if col not in categorical_features 
    and X_train[col].nunique() > 2 
    and not col.endswith(('_sin', '_cos'))
]

print("Numerical Features:", len(numerical_features), "columns")
print("Categorical Features:", categorical_features)

Numerical Features: 19 columns
Categorical Features: ['unit', 'day_of_week']


In [102]:
# Build the ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features)
    ],
    # Crucial: remainder='passthrough' keeps the remaining columns (like sin, cos, and weekend) untouched
    remainder='passthrough' 
)

In [103]:
# Apply the transformations to the training data
X_train_processed = preprocessor.fit_transform(X_train)

In [104]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_jobs=-1),
    "XGBoost": MultiOutputRegressor(XGBRegressor(n_jobs=-1)), 
    "SVM": MultiOutputRegressor(SVR())
}


NameError: name 'MultiOutputRegressor' is not defined

In [ ]:
def compute_cross_validation(model, X_train, y_train, cv_folds=5):
    """Run cross-validation and return mean and std of MAE scores across all 15 horizons."""
    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv_folds,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )
    mae_scores = -cv_scores
    return mae_scores.mean(), mae_scores.std()

   # Evaluation Loop

results = []

print("Starting Cross-Validation for 1-15 minute forecasting...\n")
for name, model in models.items():
    print(f"Evaluating {name}...")
    
    cv_mean, cv_std = compute_cross_validation(model, X_train_processed, y_train)  
    
    metrics = {
        "model_name": name,
        "cv_mae_mean": cv_mean,
        "cv_mae_std": cv_std
    }  
    results.append(metrics)  
    print(f"Average MAE ({name}): {cv_mean:.4f} (± {cv_std:.4f})\n")

# Find the best model
best_model_info = min(results, key=lambda x: x["cv_mae_mean"])
print(f" Best Model: {best_model_info['model_name']} with Average 15-min MAE: {best_model_info['cv_mae_mean']:.4f}")

Evaluating Linear Regression...
Average MAE (Linear Regression): 6.8935 (± 3.4832)

Evaluating XGBoost...
Average MAE (XGBoost): 7.8740 (± 5.5149)

Evaluating Random Forest...
Average MAE (Random Forest): 6.9983 (± 4.7803)

Evaluating SVM...
Average MAE (SVM): 84.6989 (± 63.7894)

 Best Model: Linear Regression with Average MAE: 6.8935
